# Qa 06 transfer targets

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# QA 06: Transfer Targets

Inspect transfer-target artifacts, verify reconstruction, and visualize the transfer distributions.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from notebooks.multisource_notebook_helpers import read_yaml
from src.preprocessing.transfer_targets import reconstruct_physical_from_transfer

cfg = read_yaml("../configs/training.yaml")
pc_dir = Path(cfg["data"]["point_centric_dir"])
targets_npz = np.load(pc_dir / "point_centric_Y_targets.npz", allow_pickle=True)
metadata = json.loads((pc_dir / "point_centric_metadata.json").read_text())

print("NPZ keys:", sorted(targets_npz.files))
print("target_mode:", targets_npz["target_mode"][0] if "target_mode" in targets_npz else "physical")
print(
    "physical_target_names:",
    targets_npz["physical_target_names"] if "physical_target_names" in targets_npz else [],
)
print(
    "transfer_target_names:",
    targets_npz["transfer_target_names"] if "transfer_target_names" in targets_npz else [],
)
print(
    "reference_target_names:",
    targets_npz["reference_target_names"] if "reference_target_names" in targets_npz else [],
)

In [ ]:
sites = targets_npz["target_sites"].astype(str).tolist()
site = sites[0]
safe = "".join(ch if ch.isalnum() or ch in ("_", "-") else "_" for ch in site)

physical = targets_npz[f"Yphysical__{safe}"]
reference = targets_npz[f"Yreference__{safe}"]
transfer = targets_npz[f"Ytransfer__{safe}"]

print("sample site:", site)
print("physical sample:\n", physical[:5])
print("reference sample:\n", reference[:5])
print("transfer sample:\n", transfer[:5])

In [ ]:
reconstructed = reconstruct_physical_from_transfer(transfer, reference)
abs_err = np.abs(reconstructed - physical)
print("max reconstruction error by column:", abs_err.max(axis=0))
assert np.allclose(reconstructed[:, :2], physical[:, :2], atol=1e-5, equal_nan=True)

In [ ]:
transfer_names = ["log_hs_ratio", "tp_delta", "dir_delta_deg", "dp_delta_deg"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, idx, name in zip(axes.ravel(), range(4), transfer_names):
    ax.hist(transfer[:, idx], bins=40)
    ax.set_title(name)
plt.tight_layout()

In [ ]:
# 1. Concatenate Yphysical__/Yreference__/Ytransfer__ across all sites.
# 2. Break histograms down by site and by offshore direction sector.
# 3. Flag extreme log_hs_ratio values when ref_hs is close to zero.
# 4. Compare nearest vs weighted transfer-reference behavior when both are available.